# WarehouseSort × SmolVLA — Colab T4 추론 재현본

이 노트북은 **학습이 끝난 원본 30K 체크포인트**를 GitHub에서 내려받아 Colab T4에서
Easy / Medium / Hard RGB rollout과 MP4 영상을 재현합니다.

- 사용 모델: VLM adaptation 전 `train_expert_only=true`, `freeze_vision_encoder=true` 30K
- 로컬 공식 결과: Easy 6/6, Medium 8/12, Hard 1/18, weighted 42.78%
- Kaggle 데이터는 추론에 필요하지 않습니다.
- 데이터 생성·학습 코드는 마지막에 참고용 주석으로만 남겨 두었습니다.


## 0. Colab 설정

`런타임 → 런타임 유형 변경 → T4 GPU`를 선택한 다음 아래 셀부터 실행하세요.
GitHub에 체크포인트를 올린 뒤 `CHECKPOINT_REPO`와 `CHECKPOINT_SUBDIR`만 수정하면 됩니다.
1.2GB `model.safetensors`는 일반 Git 파일이 아니라 **Git LFS**로 올려야 합니다.


In [ ]:
# ===== 사용자가 수정할 값 =====
CHECKPOINT_REPO = "https://github.com/Ahnseongmin1749/Marso-Hack-Berlin-2026.git"
CHECKPOINT_SUBDIR = "checkpoints/smolvla_all_recovery_30k"

# 빠른 1회 확인: ["easy"], 전체 재현: ["easy", "medium", "hard"]
LEVELS = ["easy", "medium", "hard"]
SEEDS = [42, 43, 44]

assert CHECKPOINT_REPO.endswith(".git"), "CHECKPOINT_REPO에 GitHub clone URL을 입력하세요."


## 1. 패키지 설치

기존 로컬 실행과 동일한 핵심 버전을 사용합니다. 설치가 끝나면 Colab 메뉴에서
**런타임 → 세션 다시 시작**을 한 번 누르고 2번 셀부터 계속하세요.


In [ ]:
%pip uninstall -y -q gradio gradio-client transformers huggingface-hub lerobot
%pip install -q --no-cache-dir --force-reinstall \
    torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 \
    --index-url https://download.pytorch.org/whl/cu128
%pip install -q --no-cache-dir \
    mani-skill==3.0.1 gymnasium==1.3.0 hydra-core==1.3.3 imageio-ffmpeg
%pip install -q --no-cache-dir "lerobot[smolvla]==0.4.4"
!apt-get -qq update && apt-get -qq install -y git-lfs
!git lfs install
print("설치 완료 — 런타임을 다시 시작한 뒤 2번부터 실행하세요.")


## 2. GPU 및 라이브러리 확인

In [ ]:
from importlib.metadata import version
import torch

print("torch:", torch.__version__)
print("lerobot:", version("lerobot"))
print("transformers:", version("transformers"))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."


## 3. WarehouseSort 코드 준비

평가 당시 사용한 대회 저장소 commit으로 고정합니다.


In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO = Path("/content/berlin-marso-hackathon")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/marso-robotics/berlin-marso-hackathon.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "6048f33217f26ae39009a812f53c81171517f393"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"], check=True)
os.chdir(REPO)

os.environ["DISPLAY"] = ""
os.environ["PYOPENGL_PLATFORM"] = "egl"
sys.path.insert(0, str(REPO))
import warehouse_sort
print("WarehouseSort 준비 완료")


## 4. GitHub에서 체크포인트 다운로드

private repository라면 clone 전에 Colab secret 또는 Git credential 설정이 필요합니다.
Git LFS가 `model.safetensors`의 실제 내용을 받았는지도 다음 셀에서 검사합니다.


In [ ]:
from pathlib import Path
import subprocess, os

CKPT_REPO_DIR = Path("/content/smolvla-checkpoint-repo")
if not CKPT_REPO_DIR.exists():
    subprocess.run(["git", "clone", CHECKPOINT_REPO, str(CKPT_REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(CKPT_REPO_DIR), "lfs", "pull"], check=True)

SMOLVLA_CKPT = CKPT_REPO_DIR / CHECKPOINT_SUBDIR
print("checkpoint:", SMOLVLA_CKPT)


## 5. 체크포인트 무결성 확인

추론에는 optimizer/training state가 필요 없지만 아래 여섯 파일은 모두 필요합니다.


In [ ]:
required = [
    "config.json",
    "model.safetensors",
    "policy_preprocessor.json",
    "policy_preprocessor_step_5_normalizer_processor.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor_step_0_unnormalizer_processor.safetensors",
]
for name in required:
    p = SMOLVLA_CKPT / name
    assert p.exists(), f"누락 파일: {p}"
    print(f"{name:70s} {p.stat().st_size / 1024**2:9.2f} MiB")

model_file = SMOLVLA_CKPT / "model.safetensors"
assert model_file.stat().st_size > 1_000_000_000, "Git LFS pointer만 받은 것 같습니다. git lfs pull을 확인하세요."
print("체크포인트 무결성 확인 완료")


## 6. SmolVLA 로드

체크포인트에 저장된 normalization과 unnormalization processor도 함께 로드합니다.


In [ ]:
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

device = torch.device("cuda")
policy = SmolVLAPolicy.from_pretrained(str(SMOLVLA_CKPT)).to(device).eval()
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config,
    pretrained_path=str(SMOLVLA_CKPT),
    preprocessor_overrides={"device_processor": {"device": "cuda"}},
)
print("SmolVLA 로드 완료")
print("n_action_steps:", policy.config.n_action_steps)


## 7. Easy / Medium / Hard 공식 방식 rollout

각 episode는 약 10초 분량입니다. 공식 evaluator와 동일하게 TimeLimit reset 직전인
`max_episode_steps - 1`회 action 뒤 상태를 판정합니다. 결과 MP4는 `/content/smolvla_eval/`에 저장됩니다.


In [ ]:
import json
import numpy as np
from pathlib import Path
from lerobot.policies.utils import prepare_observation_for_inference
from warehouse_sort.utils import compose_cfg, make_env

TASK = "Sort each parcel into the bin that matches the color of the tag on top of the parcel."
VIDEO_ROOT = Path("/content/smolvla_eval")
summary = []

for level in LEVELS:
    cfg = compose_cfg([f"difficulty={level}"])
    for seed in SEEDS:
        torch.manual_seed(seed)
        video_dir = VIDEO_ROOT / level / f"seed_{seed}"
        video_dir.mkdir(parents=True, exist_ok=True)
        env, _ = make_env(cfg, "rgb", cfg.randomization, num_envs=1, video_dir=str(video_dir))
        obs, _ = env.reset(seed=[seed])
        policy.reset(); preprocessor.reset(); postprocessor.reset()
        max_sorted = 0.0

        for _ in range(int(cfg.max_episode_steps) - 1):
            frame = prepare_observation_for_inference(
                {
                    "observation.images.scene": obs["rgb"][0].detach().cpu().numpy(),
                    "observation.state": obs["state"][0].detach().cpu().numpy().astype(np.float32),
                },
                device=device, task=TASK, robot_type="franka_panda",
            )
            with torch.inference_mode():
                action = postprocessor(policy.select_action(preprocessor(frame)))[..., :4]
            obs, _, _, _, info = env.step(action.clamp(-1, 1).to(obs["state"].device))
            if "success_count" in info:
                value = info["success_count"]
                max_sorted = max(max_sorted, float(value[0].item()))

        final_sorted = float(env.unwrapped.evaluate()["success_count"][0].item())
        sorted_count = max(max_sorted, final_sorted)
        video = video_dir / "0.mp4"
        summary.append({"level": level, "seed": seed, "sorted": sorted_count,
                        "total": int(cfg.difficulty.num_parcels), "video": str(video)})
        print(f"{level:6s} seed={seed}: {sorted_count:.0f}/{cfg.difficulty.num_parcels}  {video}")
        env.close()

(VIDEO_ROOT / "summary.json").write_text(json.dumps(summary, indent=2))
print("평가 완료:", VIDEO_ROOT / "summary.json")


## 8. 점수 집계

In [ ]:
weights = {"easy": 0.2, "medium": 0.3, "hard": 0.5}
totals = {level: sum(x["sorted"] for x in summary if x["level"] == level) for level in LEVELS}
denoms = {level: sum(x["total"] for x in summary if x["level"] == level) for level in LEVELS}
rates = {level: totals[level] / denoms[level] for level in LEVELS}

for level in LEVELS:
    print(f"{level:6s}: {totals[level]:.0f}/{denoms[level]} = {rates[level]:.2%}")
if set(LEVELS) == {"easy", "medium", "hard"}:
    weighted = sum(weights[level] * rates[level] for level in weights)
    print(f"weighted score: {weighted:.2%}")


## 9. 생성된 영상 보기

드롭다운에서 rollout을 골라 재생합니다.


In [ ]:
from IPython.display import Video, display
import ipywidgets as widgets

options = [(f"{x['level']} seed {x['seed']} — {x['sorted']:.0f}/{x['total']}", x["video"]) for x in summary]
picker = widgets.Dropdown(options=options, description="rollout:")
output = widgets.Output()

def show_video(change=None):
    with output:
        output.clear_output(wait=True)
        display(Video(picker.value, embed=True, width=800))

picker.observe(show_video, names="value")
display(picker, output)
show_video()


## 10. 결과 다운로드

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/smolvla_eval", "zip", VIDEO_ROOT)
print(archive)
# 필요할 때 다음 줄의 주석을 해제하세요.
# files.download(archive)


## 11. 학습 부분 — T4 추론본에서는 비활성화

아래 코드는 실행 기록과 설정 참고용입니다. 이 Colab 노트북에서는 데이터 600 episodes 변환,
recovery 수집, 30K 학습을 실행하지 않습니다. 체크포인트는 GitHub에서 가져옵니다.


In [ ]:
# 실행하지 않음: 대용량 Kaggle 데이터 다운로드 및 LeRobot 변환
# !python prepare_multilevel_recovery_dataset.py

# 실행하지 않음: V100에서 수행한 원본 30K action-expert 학습 설정
# !python -m lerobot.scripts.lerobot_train \
#   --dataset.repo_id=local/warehouse_smolvla_all_recovery \
#   --dataset.root=/content/warehouse_smolvla_all_recovery \
#   --policy.type=smolvla \
#   --policy.pretrained_path=lerobot/smolvla_base \
#   --policy.train_expert_only=true \
#   --policy.freeze_vision_encoder=true \
#   --policy.chunk_size=16 \
#   --policy.n_action_steps=1 \
#   --batch_size=8 --steps=30000 --optimizer.lr=1e-4

print("학습 셀은 의도적으로 주석 처리되어 있습니다.")


## 재현 기준

동일 체크포인트·환경 commit·seed를 사용한 기존 결과는 Easy 6/6, Medium 8/12, Hard 1/18입니다.
GPU 연산의 미세한 비결정성 때문에 동작 궤적이 완전히 byte-identical하지는 않을 수 있습니다.
이 노트북은 Colab에서 실제로 실행한 셀 출력과 영상이 `.ipynb`에 남도록 구성했습니다.
